#Teste AB com Parada Otimizada

Quando Usar Cada Abordagem?
 * O’Brien-Fleming (OBF):
   * Quando usar: Na grande maioria dos testes de produto e experimentação digital.
   * Por que: Ele é conservador no início (exige um Z-score muito alto para parar logo nos primeiros dias) e vai afrouxando os limites até chegar ao valor padrão no final. Isso preserva quase todo o poder estatístico (evita falsos negativos em efeitos pequenos), mas ainda permite parar cedo se o resultado for um "home run" (efeito muito grande).
 * Pocock:
   * Quando usar: Em testes com alto custo operacional/financeiro (ex: campanhas de anúncios caras, ensaios clínicos, custos diários elevados de infraestrutura) ou em fases estritamente exploratórias.
   * Por que: Ele usa limiares constantes ao longo de todas as análises. É mais rápido para interromper experimentos perdedores ou inconclusivos mais cedo, mas cobra o preço de reduzir o poder estatístico se o efeito real for marginal.


Aqui está o código adaptado. Ele une a lógica original de cálculo de fronteiras (O'Brien-Fleming e Pocock) com a reamostragem via Bootstrap para calcular a estatística Z de forma empírica, eliminando a dependência estrita da suposição de normalidade nos dados (ideal para métricas com distribuições assimétricas ou caudas longas, como receita por usuário).


In [1]:
import numpy as np
from scipy import stats

def calcular_limites_sequenciais(K, alpha=0.05):
    """
    Calcula os limites Z teóricos para O'Brien-Fleming e Pocock
    para K olhares planejados (bicaudal).
    """
    z_crit = stats.norm.ppf(1 - alpha / 2)
    k_values = np.arange(1, K + 1)

    # 1. O'Brien-Fleming: Rigoroso no início, brando no fim
    obf_boundaries = z_crit * np.sqrt(K / k_values)

    # 2. Pocock: Limites constantes ajustados para K olhares
    pocock_fator = 1.0 + 0.20 * np.log(K)
    pocock_boundaries = np.full(K, z_crit * pocock_fator)

    return k_values, obf_boundaries, pocock_boundaries

def calcular_z_bootstrap(controle, tratamento, n_boot=500):
    """
    Calcula o Z-score empírico usando Bootstrap para dados não-normais.
    """
    # Diferença observada nas amostras atuais
    obs_diff = np.mean(tratamento) - np.mean(controle)

    # Concatenação para reamostragem sob a hipótese nula ou geral
    combined = np.concatenate([controle, tratamento])
    n_c = len(controle)

    diffs = []
    for _ in range(n_boot):
        sample = np.random.choice(combined, size=len(combined), replace=True)
        boot_c = sample[:n_c]
        boot_t = sample[n_c:]
        diffs.append(np.mean(boot_t) - np.mean(boot_c))

    # Erro padrão empírico via Bootstrap
    se_boot = np.std(diffs, ddof=1)

    if se_boot == 0:
        return 0.0

    # Z-score empírico
    return obs_diff / se_boot

def simular_teste_sequencial_nao_normal(dados_controle_total, dados_tratamento_total, K=5, metodo='obf'):
    """
    Simula o teste sequencial acumulando dados ao longo de K olhares
    e avaliando a significância com Bootstrap.
    """
    _, obf_limites, pocock_limites = calcular_limites_sequenciais(K)
    limites_ativos = obf_limites if metodo.lower() == 'obf' else pocock_limites

    # Divide os dados totais em K partes (simulando o acúmulo temporal)
    tamanho_passo_c = len(dados_controle_total) // K
    tamanho_passo_t = len(dados_tratamento_total) // K

    print(f"\n--- INICIANDO SIMULAÇÃO SEQUENCIAL ({metodo.upper()}) COM BOOTSTRAP ---")

    for look in range(1, K + 1):
        # Dados acumulados até o olhar atual
        fim_c = look * tamanho_passo_c if look < K else len(dados_controle_total)
        fim_t = look * tamanho_passo_t if look < K else len(dados_tratamento_total)

        sub_controle = dados_controle_total[:fim_c]
        sub_tratamento = dados_tratamento_total[:fim_t]

        # Calcula o Z-score empírico via Bootstrap para o lote atual
        z_empirico = calcular_z_bootstrap(sub_controle, sub_tratamento, n_boot=300)
        limite_atual = limites_ativos[look - 1]

        print(f"Olhar {look}/{K} (n_c={len(sub_controle)}, n_t={len(sub_tratamento)}): "
              f"Z_boot = {z_empirico:.2f} | Limite Exigido = {limite_atual:.2f}")

        if z_empirico >= limite_atual:
            print(f"-> Efeito significativo detectado! Interrompendo o teste no olhar {look}.")
            return "Parou (Sucesso)"

    print("-> Teste concluído sem cruzamento de fronteira. Atingiu o horizonte final.")
    return "Horizonte Final"

# ==========================================
# Exemplo Prático com Dados Não-Normais (Ex: Exponenciais / Receita)
# ==========================================
np.random.seed(42)

# Gerando dados altamente assimétricos (ex: distribuição exponencial simulando receita)
controle_pop = np.random.exponential(scale=10.0, size=5000)
tratamento_pop = np.random.exponential(scale=11.5, size=5000) # Efeito real presente

# Rodando a simulação sequencial com O'Brien-Fleming e Bootstrap
simular_teste_sequencial_nao_normal(controle_pop, tratamento_pop, K=5, metodo='obf')


--- INICIANDO SIMULAÇÃO SEQUENCIAL (OBF) COM BOOTSTRAP ---
Olhar 1/5 (n_c=1000, n_t=1000): Z_boot = 3.74 | Limite Exigido = 4.38
Olhar 2/5 (n_c=2000, n_t=2000): Z_boot = 3.88 | Limite Exigido = 3.10
-> Efeito significativo detectado! Interrompendo o teste no olhar 2.


'Parou (Sucesso)'